In [3]:
# ============================================
# 01_excel_ingestion.ipynb
# Reads supplier_prices.xlsx and exports clean raw data
# ============================================

from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
import os
import pandas as pd
import json

# --- Setup shared project directory ---
PROJECT_ROOT = "/content/drive/MyDrive/project"
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/data", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/output", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/quotations", exist_ok=True)

# --- Step 1: Upload the Excel file from your system ---
print("Please select your supplier_prices.xlsx file:")
uploaded = files.upload()
excel_filename = list(uploaded.keys())[0]

# --- Step 2: Read the Excel file ---
supplier_df = pd.read_excel(excel_filename)
print(f"\n✅ Loaded {len(supplier_df)} rows from {excel_filename}")
print(f"Columns found: {list(supplier_df.columns)}")

# --- Step 3: Check expected columns are present ---
expected_cols = {"Material", "Specification", "Supplier", "Unit Price",
                  "Currency", "Unit", "Lead Time", "MOQ", "Available Stock"}
actual_cols = set(supplier_df.columns)
missing = expected_cols - actual_cols
extra = actual_cols - expected_cols

if missing:
    print(f"⚠️ Missing expected columns: {missing}")
if extra:
    print(f"ℹ️ Extra/unexpected columns found: {extra}")
if not missing:
    print("✅ All expected columns present")

# --- Step 4: Preview the data ---
print("\nFirst 5 rows:")
display(supplier_df.head())

print("\nNull value counts per column:")
print(supplier_df.isnull().sum())

print("\nRow count check:", "✅ ~30 rows as expected" if 25 <= len(supplier_df) <= 35
      else f"⚠️ Expected ~30 rows, got {len(supplier_df)}")

# --- Step 5: Basic dtype sanity check ---
if "Unit Price" in supplier_df.columns:
    non_numeric_prices = supplier_df[pd.to_numeric(supplier_df["Unit Price"], errors="coerce").isnull()]
    if len(non_numeric_prices) > 0:
        print(f"\n⚠️ {len(non_numeric_prices)} rows have non-numeric Unit Price:")
        display(non_numeric_prices)

# --- Step 6: Export raw data as JSON for downstream notebooks ---
supplier_records = supplier_df.to_dict(orient="records")

with open(f"{PROJECT_ROOT}/output/supplier_prices_raw.json", "w") as f:
    json.dump(supplier_records, f, indent=2, default=str)

print(f"\n✅ Saved {len(supplier_records)} supplier records to {PROJECT_ROOT}/output/supplier_prices_raw.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Please select your supplier_prices.xlsx file:


Saving supplier_prices.xlsx to supplier_prices (1).xlsx

✅ Loaded 30 rows from supplier_prices (1).xlsx
Columns found: ['Material', 'Specification', 'Supplier', 'Unit Price', 'Currency', 'Unit', 'Lead Time (Days)', 'MOQ', 'Available Stock']
⚠️ Missing expected columns: {'Lead Time'}
ℹ️ Extra/unexpected columns found: {'Lead Time (Days)'}

First 5 rows:


,Material,Specification,Supplier,Unit Price,Currency,Unit,Lead Time (Days),MOQ,Available Stock
0,SS304 Stainless Steel Pipe,ASTM A312,Tata Steel,820,INR,kg,10,100,5000
1,SS316 Stainless Steel Pipe,ASTM A312,Jindal Stainless,940,INR,kg,12,100,3200
2,Mild Steel Plate,IS 2062,JSW Steel,65,INR,kg,8,200,10000
3,Carbon Steel Rod,ASTM A36,SAIL,78,INR,kg,11,150,8500
4,Aluminium Sheet,AA6061,Vedanta,260,INR,kg,7,100,4200



Null value counts per column:
Material            0
Specification       0
Supplier            0
Unit Price          0
Currency            0
Unit                0
Lead Time (Days)    0
MOQ                 0
Available Stock     0
dtype: int64

Row count check: ✅ ~30 rows as expected

✅ Saved 30 supplier records to /content/drive/MyDrive/project/output/supplier_prices_raw.json
